In [262]:
import os

os.environ["PYSPARK_PYTHON"] = "python"
os.environ["PYSPARK_DRIVER_PYTHON"] = "python"

# STEP 1: SETUP PYSPARK JOB

## Basic Spark Setup

In [263]:
from pyspark.sql import SparkSession

def create_spark_session():
    return SparkSession.builder \
        .appName("SalaryDetectionJob") \
        .config("spark.python.worker.timeout", "120") \
        .config("spark.executor.heartbeatInterval", "60s") \
        .getOrCreate()

## Entry point

In [264]:
spark = create_spark_session()

df = spark.read.csv(
        "C:/Users/ACER/Downloads/salary-detector/data/test_scenarios.csv",
        header=True,
        inferSchema=True
    )

df.show(5)

+--------------------+----------+-------------------+------+------+-------+--------+
|          CustomerId|      Date|Transaction Details|  Type|Amount|Balance|Category|
+--------------------+----------+-------------------+------+------+-------+--------+
|# C1 → Normal mon...|      NULL|               NULL|  NULL|  NULL|   NULL|    NULL|
|                  C1|2024-09-05|NEFT/TCS SALARY SEP|Credit| 60000|  70000|  Salary|
|                  C1|2024-10-05|NEFT/TCS SALARY OCT|Credit| 60500| 130000|  Salary|
|                  C1|2024-11-05|NEFT/TCS SALARY NOV|Credit| 60000| 190000|  Salary|
|                  C1|2024-11-07|         UPI/Swiggy| Debit|   500| 189500|    Food|
+--------------------+----------+-------------------+------+------+-------+--------+
only showing top 5 rows


# STEP 2: CLEAN & STANDARDIZE DATA

In [265]:
from pyspark.sql.functions import col, to_date

df = df.withColumn("Date", to_date(col("Date"), "yyyy-MM-dd"))

df = df.withColumn("Amount", col("Amount").cast("double"))
df = df.withColumn("Balance", col("Balance").cast("double"))

df = df.filter(col("Date").isNotNull())

# STEP 3: FILTER CREDIT TRANSACTIONS

## PySpark version:

In [266]:
from pyspark.sql.functions import upper

credit_df = df.filter(
    (upper(col("Type")) == "CREDIT") &
    (col("Amount") >= 3000)
)

In [267]:
all_customers_df = df.select("CustomerId").distinct()

# STEP 4: SENDER EXTRACTION

In [268]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

def extract_sender(details):
    if not details:
        return "UNKNOWN"

    details = details.upper()
    parts = details.split("/")

    if len(parts) >= 2:
        sender = parts[1]
        words = sender.split()

        sender = " ".join(words[:2])

        if sender.replace(" ", "").isdigit():
            return f"ANON_{sender.replace(' ', '')}"

        return sender.strip()

    return "UNKNOWN"

extract_sender_udf = udf(extract_sender, StringType())

In [269]:
credit_df = credit_df.withColumn("sender", col("Transaction Details"))

# STEP 5: MERGE SAME-DAY TRANSACTIONS

In [270]:
from pyspark.sql.functions import sum as spark_sum

daily_df = credit_df.groupBy(
    "CustomerId", "sender", "Date"
).agg(
    spark_sum("Amount").alias("daily_amount")
)

In [271]:
from pyspark.sql.functions import collect_list

# STEP 6: COMPUTE INTERVALS

In [272]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, datediff

window_spec = Window.partitionBy("CustomerId", "sender").orderBy("Date")

daily_df = daily_df.withColumn(
    "prev_date",
    lag("Date").over(window_spec)
)

daily_df = daily_df.withColumn(
    "interval_days",
    datediff(col("Date"), col("prev_date"))
)

# STEP 7: AGGREGATE FEATURES

## Feature aggregation:

In [273]:
from pyspark.sql.functions import collect_list, avg, stddev, count

features_df = daily_df.groupBy(
    "CustomerId", "sender"
).agg(
    avg("interval_days").alias("interval_mean"),
    stddev("interval_days").alias("interval_std"),
    avg("daily_amount").alias("amount_mean"),
    stddev("daily_amount").alias("amount_std"),
    count("*").alias("count"),
    collect_list("daily_amount").alias("salary_history")  # ✅ THIS
)

In [274]:
best_df = ranked_df.filter(col("rank") == 1)

In [275]:
final_df = all_customers_df.join(
    best_df,
    on="CustomerId",
    how="left"
)

In [276]:
from pyspark.sql.functions import when

features_df = features_df.withColumn(
    "salary_cycle",
    when(col("interval_mean").isNull(), "INSUFFICIENT_DATA")
    .when((col("interval_mean") >= 25) & (col("interval_mean") <= 35), "MONTHLY")
    .when((col("interval_mean") >= 5) & (col("interval_mean") <= 9), "WEEKLY")
    .when((col("interval_mean") >= 12) & (col("interval_mean") <= 18), "BI-WEEKLY")
    .otherwise("IRREGULAR")
)

In [277]:
from pyspark.sql.functions import col, lower

# 1. periodic check
features_df = features_df.withColumn(
    "is_periodic",
    (
        col("interval_mean").isNotNull() &
        (
            col("interval_mean").between(24, 35) |   # monthly
            col("interval_mean").between(5, 10)  |   # weekly
            col("interval_mean").between(10, 22)     # semi-monthly / flexible
        )
    )
)

# 2. stable amount
features_df = features_df.withColumn(
    "is_stable_amount",
    col("amount_std") < 10000
)

# 3. repetition
features_df = features_df.withColumn(
    "has_repetition",
    col("count") >= 2
)

features_df = features_df.withColumn(
    "has_salary_keyword",
    lower(col("sender")).rlike("salary|sal|payroll")
)

In [278]:
from pyspark.sql.functions import when

features_df = features_df.withColumn(
    "salary_confidence",
    when(col("count") >= 3, "HIGH")
    .when(col("count") == 2, "MEDIUM")
    .otherwise("LOW")
)

In [279]:
features_df = features_df.withColumn(
    "is_salary_account",
    (
        col("is_periodic") &
        col("is_stable_amount") &
        col("has_repetition") &
        (
            col("has_salary_keyword") |
            (col("amount_mean") > 30000)
        )
    )
)

In [280]:
from pyspark.sql.functions import sort_array

features_df = features_df.withColumn(
    "salary_history",
    sort_array(col("salary_history"))
)

In [281]:
from pyspark.sql.functions import coalesce, lit

features_df = features_df.withColumn(
    "interval_std",
    coalesce(col("interval_std"), lit(0.0))
)

features_df = features_df.withColumn(
    "amount_std",
    coalesce(col("amount_std"), lit(0.0))
)

features_df = features_df.withColumn(
    "amount_mean",
    coalesce(col("amount_mean"), lit(1.0))  # avoid division issues
)

# STEP 8: APPLY SALARY RULES

In [282]:
features_df = features_df.withColumn(
    "is_periodic",
    (
        col("interval_mean").isNotNull() &
        (
            col("interval_mean").between(24, 35) |   # monthly
            col("interval_mean").between(5, 10)  |   # weekly
            col("interval_mean").between(10, 22)     # semi-monthly / flexible
        )
    )
)

features_df = features_df.withColumn(
    "is_stable_amount",
    col("amount_std") < 10000
)

features_df = features_df.withColumn(
    "has_repetition",
    col("count") >= 2
)


In [283]:
df.filter(col("CustomerId") == "C4").show(truncate=False)

+----------+----------+-------------------+------+------+-------+--------+
|CustomerId|Date      |Transaction Details|Type  |Amount|Balance|Category|
+----------+----------+-------------------+------+------+-------+--------+
|C4        |2024-09-10|Cashback Offer     |Credit|500.0 |1000.0 |Reward  |
|C4        |2024-10-10|Cashback Offer     |Credit|600.0 |1600.0 |Reward  |
|C4        |2024-11-10|Cashback Offer     |Credit|550.0 |2150.0 |Reward  |
+----------+----------+-------------------+------+------+-------+--------+



# STEP 9: SCORING

## Score calculation:

In [284]:
features_df = features_df.withColumn(
    "time_score",
    1 / (1 + col("interval_std"))
)

features_df = features_df.withColumn(
    "amount_score",
    1 / (1 + (col("amount_std") / (col("amount_mean") + 1)))
)

features_df = features_df.withColumn(
    "salary_boost",
    (col("amount_mean") > 30000).cast("int")
)

features_df = features_df.withColumn(
    "count_score",
    (col("count") / 5)
)

features_df = features_df.withColumn(
    "final_score",
    (
        0.5 * col("time_score") +
        0.25 * col("amount_score") +
        0.15 * col("count_score") +
        0.1 * col("salary_boost")
    )
)

# STEP 10: PICK BEST SENDER PER CUSTOMER

## Window ranking

In [285]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

rank_window = Window.partitionBy("CustomerId").orderBy(col("final_score").desc())

ranked_df = features_df.withColumn(
    "rank",
    row_number().over(rank_window)
)

best_df = ranked_df.filter(col("rank") == 1)

In [286]:
final_df.printSchema()

root
 |-- CustomerId: string (nullable = true)
 |-- sender: string (nullable = true)
 |-- interval_mean: double (nullable = true)
 |-- interval_std: double (nullable = true)
 |-- amount_mean: double (nullable = true)
 |-- amount_std: double (nullable = true)
 |-- count: long (nullable = true)
 |-- salary_history: array (nullable = true)
 |    |-- element: double (containsNull = false)
 |-- salary_cycle: string (nullable = true)
 |-- is_periodic: boolean (nullable = true)
 |-- is_stable_amount: boolean (nullable = true)
 |-- has_repetition: boolean (nullable = true)
 |-- has_salary_keyword: boolean (nullable = true)
 |-- salary_confidence: string (nullable = true)
 |-- is_salary_account: boolean (nullable = true)
 |-- time_score: double (nullable = true)
 |-- amount_score: double (nullable = true)
 |-- salary_boost: integer (nullable = true)
 |-- count_score: double (nullable = true)
 |-- final_score: double (nullable = true)
 |-- rank: integer (nullable = true)



## STEP 11: OUTPUT

In [287]:
features_df.select(
    "CustomerId",
    "sender",
    "salary_cycle",
    "count",
    "interval_mean",
    "amount_mean",
    "is_periodic",
    "is_stable_amount",
    "has_repetition",
    "has_salary_keyword",
    "salary_confidence"
).show(truncate=False)

+----------+-------------------+-----------------+-----+-------------+------------------+-----------+----------------+--------------+------------------+-----------------+
|CustomerId|sender             |salary_cycle     |count|interval_mean|amount_mean       |is_periodic|is_stable_amount|has_repetition|has_salary_keyword|salary_confidence|
+----------+-------------------+-----------------+-----+-------------+------------------+-----------+----------------+--------------+------------------+-----------------+
|C1        |NEFT/TCS SALARY NOV|INSUFFICIENT_DATA|1    |NULL         |60000.0           |false      |true            |false         |true              |LOW              |
|C1        |NEFT/TCS SALARY OCT|INSUFFICIENT_DATA|1    |NULL         |60500.0           |false      |true            |false         |true              |LOW              |
|C1        |NEFT/TCS SALARY SEP|INSUFFICIENT_DATA|1    |NULL         |60000.0           |false      |true            |false         |true        

In [289]:
from pyspark.sql.functions import col, when, lit, concat_ws, format_number

final_output = final_df.select(
    col("CustomerId"),

    # sender
    when(col("sender").isNull(), "NO_SALARY_DETECTED")
    .otherwise(col("sender"))
    .alias("sender"),

    # salary history (array → string)
    when(col("salary_history").isNull(), "-")
    .otherwise(concat_ws(", ", col("salary_history")))
    .alias("salary_history"),

    # avg salary (numeric safe)
    when(col("amount_mean").isNull(), "-")
    .otherwise(format_number(col("amount_mean"), 0))
    .alias("avg_salary"),

    # cycle
    when(col("salary_cycle").isNull(), "NO_PATTERN")
    .otherwise(col("salary_cycle"))
    .alias("salary_cycle"),

    # score
    when(col("final_score").isNull(), "0.00")
    .otherwise(format_number(col("final_score"), 2))
    .alias("score"),

    # confidence
    when(col("salary_confidence").isNull(), "NONE")
    .otherwise(col("salary_confidence"))
    .alias("salary_confidence"),

    # flag (boolean stays boolean)
    when(col("is_salary_account").isNull(), False)
    .otherwise(col("is_salary_account"))
    .alias("is_salary_account"),

    # reason (boolean logic works properly now)
    when(col("sender").isNull(), "NO SALARY SIGNAL")
    .otherwise(
        concat_ws(" | ",
            when(col("is_periodic"), "Periodic").otherwise("Not Periodic"),
            when(col("is_stable_amount"), "Stable Amount").otherwise("Variable"),
            when(col("has_repetition"), "Repeated").otherwise("Not Repeated"),
            when(col("has_salary_keyword"), "Keyword").otherwise("No Keyword")
        )
    ).alias("reason")
)

final_output.show(truncate=False)

DateTimeException: [CANNOT_PARSE_TIMESTAMP] Text ' NO company → pattern-based detection)' could not be parsed at index 0. Use `try_to_date` to tolerate invalid input string and return NULL instead. SQLSTATE: 22007